In [1]:
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, HTML
import json

In [2]:
google_api_key = os.getenv('GOOGLE_API_KEY')

groq_api_key = os.getenv('GROQ_API_KEY')

In [3]:
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")


if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

Google API Key exists and begins AI
Groq API Key exists and begins gsk_


In [4]:
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

groq_url = "https://api.groq.com/openai/v1"

In [5]:
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [6]:
debugger_prompt = """You are a strict snarky debugger discussing with a Creative Solver and Documentation Expert.
- Only talk to the other agents, never the user
- Only use information already in the conversation
- Keep responses short and technical
- Add something new each round, no repetition
- Also identify the most likely root cause.
- Respond in plain text only. Do NOT use JSON.
"""

creative_prompt = """You are an empathetic creative problem solver discussing with a Debugger and Documentation Expert.
- Only talk to the other agents, never the user
- Only use information already in the conversation
- Propose alternatives, challenge or build on the Debugger
- Keep responses short and solution-focused
- Add something new each round, no repetition
- Propose your fix in message if you agree, and any alternative fix in alternative if you challenge the narrative of others.
- Respond in plain text only. Do NOT use JSON.
"""

docs_prompt = """You are a documentation expert discussing with a Debugger and Creative Solver.
- Only talk to the other agents, never the user
- Only use information already in the conversation
- Ask max 2 clarifying questions per round
- Dont start summarizing until explicitly asked to do so
- If user says max rounds reached, summarize immediately and then give corrected_code
- if asked to briefly summarize the conversation so far, summarize immediately
- Respond in plain text only. Do NOT use JSON.
"""

In [7]:
def ask_agent(client, model, system_prompt, conversation_history):
    
    messages = [{"role": "system", "content": system_prompt}] + conversation_history
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=3000,    
    )
    raw = response.choices[0].message.content
    
    
    if not raw or not raw.strip():
            raw = "I need more context to continue."
                
    
    return raw.strip()

In [1]:
def call_debugger(conversation):
    
    response = ask_agent(gemini, "gemini-3-pro-preview", debugger_prompt, conversation)
    conversation.append({
        "role": "assistant", 
        "content": f"{response}"
    })
    
    
    return response
    

In [2]:
def call_creativesolver(conversation):
    response = ask_agent(groq, "openai/gpt-oss-20b", creative_prompt, conversation)
    
    conversation.append({
            "role": "assistant",
            "content": f"{response}"})
     
    return response

In [3]:
def call_docs(conversation, force_summary=False, interim_summary=False):
    if force_summary:
        conversation.append({
            "role": "user",
            "content": "Max rounds reached. Please write the final summary and corrected code."
        })
    elif interim_summary:
        conversation.append({
            "role": "user",
            "content": "Briefly summarize the discussion so far in 2–3 sentences."
        })

    response = ask_agent(groq, "openai/gpt-oss-20b", docs_prompt, conversation)

    conversation.append({
        "role": "assistant",
        "content": f"{response}"
    })

    
    return response

In [4]:
def display_docs(resp):
    display(HTML(f"<h3 style='color:lightgreen'>📝 [Docs Expert]</h3><p>{resp}</p>"))

def display_debugger(resp):
    display(HTML(f"<h3 style='color:coral'>🔍 [Debugger]</h3><p>{resp}</p>"))

def display_creative(resp):
    display(HTML(f"<h3 style='color:cyan'>💡 [Creative Solver]</h3><p>{resp}</p>"))

In [5]:
def run_debug_session(code_snippet, error_message):
    
    conversation = [
        {"role": "user", "content": f"Code:\n{code_snippet}\n\nError:\n{error_message}"}]
   
    display(Markdown(f"### -----MULTI-LLM DEBUG SESSION STARTED-------\n"))

    round_counter = 0
    while round_counter <5 :
        debugger_response = call_debugger(conversation)
        display_debugger(debugger_response)
        
        creative_response = call_creativesolver(conversation)
        display_creative(creative_response) 

        if round_counter == 3:
           docs_response = call_docs(conversation, force_summary=False, interim_summary = True)
        else:
            docs_response = call_docs(conversation, force_summary = False, interim_summary =False)
        display_docs(docs_response)
        
    
        
        round_counter +=1
        
    display(Markdown("### ------------Wrap Up---------"))
    docs_response = call_docs(conversation, force_summary=True, interim_summary=False)
    display_docs(docs_response)

    
    

In [214]:
err_msg = "ytmusicapi.exceptions.YTMusicUserError: Your cookie is missing the required value __Secure-3PAPISID"
code_snippet = """from ytmusicapi import YTMusic
yt = YTMusic('browser.json')"""
run_debug_session(code_snippet=code_snippet, error_message=err_msg)

### -----MULTI-LLM DEBUG SESSION STARTED-------


### ------------Wrap Up---------

In [219]:
code_snippet = """
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)

result = calculate_average([])
print(result)
"""

error_message = "ZeroDivisionError: division by zero"
run_debug_session(code_snippet=code_snippet, error_message=error_message)

### -----MULTI-LLM DEBUG SESSION STARTED-------


### ------------Wrap Up---------